In [1]:
import pandas as pd
from tqdm import tqdm
from itables import show as table_show

In [5]:
ENTREPOT_PATH = '~/Bureau/utils/data/'
METEO_PATH = '~/Bureau/utils/data/meteo/'
SPATIAL_PATH = './data/spatial/'

In [6]:
df = {}

def import_df(df_name, path_data, sep, index_col=None):
    df[df_name] = pd.read_csv(path_data+df_name+'.csv', sep = sep, index_col=index_col, low_memory=False)

def import_dfs(df_names, path_data, sep = ',', index_col=None, verbose=False):
    for df_name in tqdm(df_names) : 
        if(verbose) :
            print(" - ", df_name)
        import_df(df_name, path_data, sep, index_col=index_col)

tables_entrepot = [
    'recolte_rendement_prix',
    'recolte_rendement_prix_restructure',
    'sdc', 
    'destination_valorisation',
    'action_synthetise',
    'action_synthetise_agrege',
    'plantation_perenne_phases_synthetise', 
    'plantation_perenne_synthetise',
    'composant_culture',
    'espece'
]

tables_performances = []

# import des données de l'entrepôt avec la colonne 'id' en index 
import_dfs(tables_entrepot, ENTREPOT_PATH, sep = ',',index_col='id',verbose=False) 
import_dfs(tables_performances, ENTREPOT_PATH, sep = ',',verbose=False) 

100%|██████████| 10/10 [00:19<00:00,  1.96s/it]
0it [00:00, ?it/s]


In [7]:
studied_sdc_ids = ['fr.inra.agrosyst.api.entities.GrowingSystem_ad0f459f-d4ff-4808-83b3-18e88bfc6f30',
       'fr.inra.agrosyst.api.entities.GrowingSystem_f1121037-e652-44a6-b02b-6033c4ded08f',
       'fr.inra.agrosyst.api.entities.GrowingSystem_360a4405-2ab8-4801-964f-3e697eb98074',
       'fr.inra.agrosyst.api.entities.GrowingSystem_8af89b39-dc18-4a88-8e10-3f1cbd6fd33f',
       'fr.inra.agrosyst.api.entities.GrowingSystem_319de5fa-79d9-4467-a7b4-e56dfd4b0eeb',
       'fr.inra.agrosyst.api.entities.GrowingSystem_5be7ded2-e833-4352-a877-2d2f92ec4e06',
       'fr.inra.agrosyst.api.entities.GrowingSystem_945cccb1-7ac0-473e-a2f4-aa04147ce470',
       'fr.inra.agrosyst.api.entities.GrowingSystem_d75bf441-2054-44d1-8b97-19a5c85487d7',
       'fr.inra.agrosyst.api.entities.GrowingSystem_7530209d-d8c9-4d66-aad3-81b8ef747f0f']

In [8]:
FILIERE = "VITICULTURE"

In [9]:
# ajout des informations nécessaires à recolte_rendement_prix
left = df['recolte_rendement_prix']
right = pd.concat([df['action_synthetise_agrege'], df['action_synthetise_agrege']])
df['recolte_rendement_prix_extanded'] = pd.merge(left, right, left_on = 'action_id', right_index=True, how='left')

# ajout des informations nécessaires à recolte_rendement_prix
left = df['recolte_rendement_prix_extanded']
right = df['sdc'][['filiere']]
df['recolte_rendement_prix_extanded'] = pd.merge(left, right, left_on = 'sdc_id', right_index=True, how='left')


In [15]:
df['recolte_rendement_prix_extanded_test'] = df['recolte_rendement_prix_extanded'].loc[
    df['recolte_rendement_prix_extanded']['sdc_id'].isin(studied_sdc_ids)
]
df['recolte_rendement_prix_test'] = df['recolte_rendement_prix'].loc[
    df['recolte_rendement_prix'].index.isin(df['recolte_rendement_prix_extanded_test'].index)
]
df['recolte_rendement_prix_restructure_test'] = df['recolte_rendement_prix_restructure'].loc[
    df['recolte_rendement_prix_restructure'].index.isin(df['recolte_rendement_prix_test'].index)
]
df['sdc_test'] = df['sdc'].loc[
    df['sdc'].index.isin(studied_sdc_ids)
]
df['action_synthetise_test'] = df['action_synthetise'].loc[
    df['action_synthetise'].index.isin(df['recolte_rendement_prix_test']['action_id'])
]
df['action_synthetise_agrege_test'] = df['action_synthetise_agrege'].loc[
    df['action_synthetise_agrege'].index.isin(df['action_synthetise_test'].index)
]
df['composant_culture_test'] = df['composant_culture'].loc[
    df['composant_culture'].index.isin(df['recolte_rendement_prix_restructure_test']['composant_culture_id'])
]
df['espece_test'] = df['espece'].loc[
    df['espece'].index.isin(df['composant_culture_test']['espece_id'])
]

In [16]:
path='./'
df['recolte_rendement_prix_test'].to_csv(path+'recolte_rendement_prix.csv')
df['recolte_rendement_prix_restructure_test'].to_csv(path+'recolte_rendement_prix_restructure'+'.csv')
df['sdc_test'].to_csv(path+'sdc'+'.csv')
df['action_synthetise_test'].to_csv(path+'action_synthetise'+'.csv')
df['action_synthetise_agrege_test'].to_csv(path+'action_synthetise_agrege'+'.csv')
df['composant_culture_test'].to_csv(path+'composant_culture'+'.csv')
df['espece_test'].to_csv(path+'espece'+'.csv')

In [17]:
df['composant_culture_test']

,code,espece_id,surface_relative,variete_id,compagne,culture_id
id,,,,,,
fr.inra.agrosyst.api.entities.CroppingPlanSpecies_27417965-8e1d-413d-a444-ad60b98ca994,4886e043-8f50-40b3-9303-fe19f9dc840f,fr.inra.agrosyst.api.entities.referential.RefE...,NaN,fr.inra.agrosyst.api.entities.referential.RefV...,NaN,fr.inra.agrosyst.api.entities.CroppingPlanEntr...
fr.inra.agrosyst.api.entities.CroppingPlanSpecies_1a947103-ebce-450e-bbd8-ccb33a64f9b7,c66ce3dd-617a-436e-bdd8-99c2904de696,fr.inra.agrosyst.api.entities.referential.RefE...,NaN,fr.inra.agrosyst.api.entities.referential.RefV...,NaN,fr.inra.agrosyst.api.entities.CroppingPlanEntr...
fr.inra.agrosyst.api.entities.CroppingPlanSpecies_ec94d47b-812a-42aa-85a3-b9edff10c2cb,99c01fb0-279e-4002-ab8d-1880e61c227a,fr.inra.agrosyst.api.entities.referential.RefE...,NaN,fr.inra.agrosyst.api.entities.referential.RefV...,NaN,fr.inra.agrosyst.api.entities.CroppingPlanEntr...
fr.inra.agrosyst.api.entities.CroppingPlanSpecies_da33f038-6738-492a-951a-0f7d2af1786d,610ec793-2ea2-4ce4-83d5-0027587fb061,fr.inra.agrosyst.api.entities.referential.RefE...,NaN,fr.inra.agrosyst.api.entities.referential.RefV...,NaN,fr.inra.agrosyst.api.entities.CroppingPlanEntr...
fr.inra.agrosyst.api.entities.CroppingPlanSpecies_aa214470-6325-4008-a71a-46a7c3e11419,566a674b-f181-467a-916e-9824c2f4bca1,fr.inra.agrosyst.api.entities.referential.RefE...,NaN,fr.inra.agrosyst.api.entities.referential.RefV...,NaN,fr.inra.agrosyst.api.entities.CroppingPlanEntr...
fr.inra.agrosyst.api.entities.CroppingPlanSpecies_d580b4db-1f42-435d-a5e3-6fe62fde77c4,ecea0ae4-0f3b-4251-a0da-ef9ac163f125,fr.inra.agrosyst.api.entities.referential.RefE...,NaN,fr.inra.agrosyst.api.entities.referential.RefV...,NaN,fr.inra.agrosyst.api.entities.CroppingPlanEntr...
fr.inra.agrosyst.api.entities.CroppingPlanSpecies_16d80c65-e803-42d0-85c5-0862f93f861c,a04cc00a-15b2-458b-bb87-ea9b81a6c9e1,fr.inra.agrosyst.api.entities.referential.RefE...,NaN,fr.inra.agrosyst.api.entities.referential.RefV...,NaN,fr.inra.agrosyst.api.entities.CroppingPlanEntr...
fr.inra.agrosyst.api.entities.CroppingPlanSpecies_c0da8f3d-123b-40d1-9c44-09df270e03e2,2152b9c9-6d4a-49e0-98f8-4f8a1552dbb7,fr.inra.agrosyst.api.entities.referential.RefE...,NaN,fr.inra.agrosyst.api.entities.referential.RefV...,NaN,fr.inra.agrosyst.api.entities.CroppingPlanEntr...
fr.inra.agrosyst.api.entities.CroppingPlanSpecies_46c45ab8-e21d-465d-bc28-6b99165f7c52,bd695d1a-4db5-413c-915f-7f2c4f8f5a59,fr.inra.agrosyst.api.entities.referential.RefE...,NaN,fr.inra.agrosyst.api.entities.referential.RefV...,NaN,fr.inra.agrosyst.api.entities.CroppingPlanEntr...
